# 03 — Feature Engineering
Build daily metrics per trader, drawdown proxy, trader segments.

**Input:** `data/processed/merged_data.csv`
**Output:** `data/processed/daily_metrics.csv`, `data/processed/trader_summary.csv`

In [1]:
import sys
sys.path.append('..')
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import os
from src.config import MERGED_FILE, DATA_PROCESSED
from src.utils import print_section
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Load merged data

In [2]:
df = pd.read_csv(MERGED_FILE)
df['date'] = pd.to_datetime(df['date'])
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

Shape: (184263, 18)
Columns: ['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side', 'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL', 'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID', 'Timestamp', 'date', 'Classification']


,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp,date,Classification
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.8700,7872.1600,BUY,02-12-2024 22:50,0.0000,Buy,0.0000,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.3454,895000000000000.0000,1730000000000.0000,2024-10-27,Greed
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.0000,127.6800,BUY,02-12-2024 22:50,986.5246,Buy,0.0000,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.0056,443000000000000.0000,1730000000000.0000,2024-10-27,Greed
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.0900,1150.6300,BUY,02-12-2024 22:50,1002.5190,Buy,0.0000,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.0504,660000000000000.0000,1730000000000.0000,2024-10-27,Greed


## 2. Identify buy / sell trades

In [3]:
print('Side values    :', df['Side'].unique())
print('Direction values:', df['Direction'].unique())

df['is_buy']  = (df['Side'] == 'BUY').astype(int)
df['is_sell'] = (df['Side'] == 'SELL').astype(int)
df['is_win']  = (df['Closed PnL'] > 0).astype(int)

print(f'Buy trades  : {df["is_buy"].sum():,}')
print(f'Sell trades : {df["is_sell"].sum():,}')
print(f'Winning trades: {df["is_win"].sum():,}')

Side values    : ['BUY' 'SELL']
Direction values: ['Buy' 'Sell' 'Open Long' 'Close Long' 'Spot Dust Conversion' 'Open Short'
 'Close Short' 'Long > Short' 'Short > Long' 'Auto-Deleveraging'
 'Settlement']
Buy trades  : 88,378
Sell trades : 95,885
Winning trades: 77,455


## 3. Daily metrics per trader

In [4]:
daily_metrics = df.groupby(['date', 'Account', 'Classification']).agg(
    daily_pnl      = ('Closed PnL', 'sum'),
    num_trades     = ('Closed PnL', 'count'),
    win_rate       = ('is_win',     'mean'),
    avg_size_usd   = ('Size USD',   'mean'),
    total_size_usd = ('Size USD',   'sum'),
    avg_size_tokens= ('Size Tokens','mean'),
    avg_fee        = ('Fee',        'mean'),
    total_fee      = ('Fee',        'sum'),
    buy_trades     = ('is_buy',     'sum'),
    sell_trades    = ('is_sell',    'sum'),
    gross_pnl_pos  = ('Closed PnL', lambda x: x[x > 0].sum()),
    gross_pnl_neg  = ('Closed PnL', lambda x: x[x < 0].sum()),
).reset_index()

daily_metrics['buy_sell_ratio'] = (
    daily_metrics['buy_trades'] /
    (daily_metrics['sell_trades'] + 1e-9)
)
daily_metrics['is_profitable'] = (daily_metrics['daily_pnl'] > 0).astype(int)
daily_metrics['net_pnl_after_fee'] = daily_metrics['daily_pnl'] - daily_metrics['total_fee']

print(f'Daily metrics shape: {daily_metrics.shape}')
daily_metrics.head()

Daily metrics shape: (77, 18)


,date,Account,Classification,daily_pnl,num_trades,win_rate,avg_size_usd,total_size_usd,avg_size_tokens,avg_fee,total_fee,buy_trades,sell_trades,gross_pnl_pos,gross_pnl_neg,buy_sell_ratio,is_profitable,net_pnl_after_fee
0,2023-03-28,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,Greed,0.0000,3,0.0000,159.0000,477.0000,0.0838,0.0000,0.0000,3,0,0.0000,0.0000,3000000000.0000,0,0.0000
1,2023-11-14,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,Greed,0.0000,2,0.0000,23066.9350,46133.8700,0.5000,5.5361,11.0721,2,0,0.0000,0.0000,2000000000.0000,0,-11.0721
2,2023-11-14,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,Greed,155.5034,1043,0.2752,11034.7995,11509295.8900,2571.1323,2.6687,2783.4985,489,554,15632.6699,-15477.1665,0.8827,1,-2627.9952
3,2024-03-09,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,Greed,-5564.0161,27,0.3333,3048.5944,82312.0500,129.5278,0.1445,3.9026,4,23,152.8536,-5716.8697,0.1739,0,-5567.9187
4,2024-03-09,0x430f09841d65beb3f27765503d0f850b8bce7713,Greed,0.0000,88,0.0000,1136.3127,99995.5200,8550.4773,2.8762,253.1060,88,0,0.0000,0.0000,88000000000.0000,0,-253.1060


## 4. Drawdown proxy per trader

In [5]:
def compute_max_drawdown(pnl_series):
    cumulative  = pnl_series.cumsum()
    rolling_max = cumulative.cummax()
    drawdown    = cumulative - rolling_max
    return drawdown.min()

drawdown_df = (
    daily_metrics
    .sort_values(['Account', 'date'])
    .groupby('Account')['daily_pnl']
    .apply(compute_max_drawdown)
    .reset_index()
    .rename(columns={'daily_pnl': 'max_drawdown'})
)
print(f'Drawdown computed for {len(drawdown_df):,} accounts')
print(drawdown_df['max_drawdown'].describe())

Drawdown computed for 32 accounts
count       32.0000
mean     -3308.3639
std      11735.0173
min     -59349.6771
25%          0.0000
50%          0.0000
75%          0.0000
max          0.0000
Name: max_drawdown, dtype: float64


## 5. Trader-level summary

In [6]:
trader_summary = daily_metrics.groupby('Account').agg(
    total_pnl        = ('daily_pnl',      'sum'),
    avg_daily_pnl    = ('daily_pnl',      'mean'),
    overall_winrate  = ('win_rate',        'mean'),
    avg_trades_day   = ('num_trades',      'mean'),
    total_trades     = ('num_trades',      'sum'),
    active_days      = ('date',            'nunique'),
    avg_size_usd     = ('avg_size_usd',    'mean'),
    avg_buy_sell     = ('buy_sell_ratio',  'mean'),
    total_fees_paid  = ('total_fee',       'sum'),
).reset_index()

trader_summary = trader_summary.merge(drawdown_df, on='Account', how='left')

trader_summary['size_segment'] = pd.qcut(
    trader_summary['avg_size_usd'], q=3,
    labels=['Small', 'Mid', 'Large'], duplicates='drop'
)
trader_summary['frequency_segment'] = pd.qcut(
    trader_summary['avg_trades_day'], q=3,
    labels=['Infrequent', 'Moderate', 'Frequent'], duplicates='drop'
)
trader_summary['winner_segment'] = np.where(
    trader_summary['overall_winrate'] >= 0.55, 'Consistent Winner', 'Inconsistent'
)

print(f'Trader summary shape: {trader_summary.shape}')
print()
for seg in ['size_segment','frequency_segment','winner_segment']:
    print(f'{seg}:')
    print(trader_summary[seg].value_counts())
    print()
trader_summary.head()

Trader summary shape: (32, 14)

size_segment:
size_segment
Small    11
Large    11
Mid      10
Name: count, dtype: int64

frequency_segment:
frequency_segment
Infrequent    11
Frequent      11
Moderate      10
Name: count, dtype: int64

winner_segment:
winner_segment
Inconsistent         29
Consistent Winner     3
Name: count, dtype: int64



,Account,total_pnl,avg_daily_pnl,overall_winrate,avg_trades_day,total_trades,active_days,avg_size_usd,avg_buy_sell,total_fees_paid,max_drawdown,size_segment,frequency_segment,winner_segment
0,0x083384f897ee0f19899168e3b1bec365f52a9012,1600229.8200,800114.9100,0.2158,1909.0000,3818,2,15578.0669,0.6558,7405.3123,0.0000,Large,Moderate,Inconsistent
1,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,37706.1731,18853.0866,0.4831,1926.5000,3853,2,2093.0508,0.7953,1867.0818,0.0000,Small,Moderate,Inconsistent
2,0x271b280974205ca63b716753467d5a371de622ab,31763.0884,10587.6961,0.4102,382.0000,1146,3,16282.9203,0.9899,3689.0570,0.0000,Large,Infrequent,Inconsistent
3,0x28736f43f1e871e6aa8b1148d38d4994275d72c4,132315.4817,66157.7408,0.4382,6633.0000,13266,2,508.4140,1.0213,2216.8243,0.0000,Small,Frequent,Inconsistent
4,0x2c229d22b100a7beb69122eed721cee9b24011dd,168627.9841,84313.9920,0.5041,1617.0000,3234,2,3073.4132,0.6056,3107.2480,0.0000,Small,Moderate,Inconsistent


## 6. Save outputs

In [7]:
os.makedirs(DATA_PROCESSED, exist_ok=True)
daily_metrics.to_csv(f'{DATA_PROCESSED}/daily_metrics.csv',  index=False)
trader_summary.to_csv(f'{DATA_PROCESSED}/trader_summary.csv', index=False)
print(f'daily_metrics.csv  -> {daily_metrics.shape}')
print(f'trader_summary.csv -> {trader_summary.shape}')

daily_metrics.csv  -> (77, 18)
trader_summary.csv -> (32, 14)
